# Module 2: Databricks Examples

This notebook contains conceptual and runnable examples corresponding to Chapters 6 through 13. *Note: Some cells require a Databricks environment to execute successfully.*

## Chapter 6: Introduction to Databricks

Writing to and reading from Delta Lake.

In [ ]:
# Delta Lake Example
data = [("Alice", 28), ("Bob", 25)]
df = spark.createDataFrame(data, ["Name", "Age"])

# Note: This will create a local delta-table directory if run outside Databricks
df.write.format("delta").mode("overwrite").save("/tmp/delta-table")

df_delta = spark.read.format("delta").load("/tmp/delta-table")
df_delta.show()

## Chapter 7: Core Concepts

Connecting to external data sources (conceptual).

In [ ]:
# Connecting to ADLS Gen2 (Requires valid credentials)
# storage_account_name = "myadlsaccount"
# storage_account_key = "my_access_key"

# spark.conf.set(
#     f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
#     storage_account_key
# )

# df = spark.read.csv(f"abfss://mycontainer@{storage_account_name}.dfs.core.windows.net/data.csv")
print("Uncomment and provide credentials to test ADLS connection.")

## Chapter 8: Data Engineering

Delta Lake MERGE operations.

In [ ]:
from delta.tables import DeltaTable

# Create a target table
spark.range(5).write.format("delta").mode("overwrite").save("/tmp/target_table")
targetTable = DeltaTable.forPath(spark, "/tmp/target_table")

# Create updates DataFrame
updatesDF = spark.range(3, 7).withColumnRenamed("id", "new_id")

# Perform MERGE (Upsert)
targetTable.alias("t").merge(
    updatesDF.alias("s"),
    "t.id = s.new_id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsert(values={"id": "s.new_id"}) \
 .execute()

spark.read.format("delta").load("/tmp/target_table").show()

## Chapter 9: Machine Learning

MLflow Tracking Example.

In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.datasets import make_regression
from sklearn.metrics import mean_squared_error

# Generate synthetic data
X, y = make_regression(n_features=4, n_informative=2, random_state=0, shuffle=False)

with mlflow.start_run():
    rf = RandomForestRegressor(max_depth=2, random_state=0)
    rf.fit(X, y)
    
    preds = rf.predict(X)
    mse = mean_squared_error(y, preds)
    
    mlflow.log_param("max_depth", 2)
    mlflow.log_metric("mse", mse)
    
    print(f"Model trained with MSE: {mse}")
    # mlflow.sklearn.log_model(rf, "model") # Uncomment to log model artifact

## Chapter 11: Security & Governance

Using Databricks Secrets (conceptual).

In [ ]:
# Databricks Secrets Example (Requires Databricks environment and configured secret scope)
# try:
#     db_password = dbutils.secrets.get(scope="my-key-vault-scope", key="database-password")
#     print("Secret retrieved successfully.")
# except NameError:
#     print("dbutils is only available in a Databricks notebook environment.")
print("This cell demonstrates dbutils usage.")

## Chapter 13: Databricks Mosaic AI

Querying Foundation Models via MLflow Deployments (conceptual).

In [ ]:
# Mosaic AI Model Serving Example
# from mlflow.deployments import get_deploy_client

# try:
#     client = get_deploy_client("databricks")
#     response = client.predict(
#         endpoint="databricks-dbrx-instruct",
#         inputs={
#             "messages": [
#                 {"role": "user", "content": "Hello!"}
#             ],
#             "max_tokens": 50
#         }
#     )
#     print(response["choices"][0]["message"]["content"])
# except Exception as e:
#     print("Requires Databricks environment and active endpoint.", e)
print("This cell demonstrates querying Mosaic AI endpoints.")